In [8]:
import pandas as pd
import numpy as np
import datetime
from sunpy.net import Fido,attrs as a

import warnings
warnings.filterwarnings('ignore')

In [9]:
# Change the forecast window according to the data you want to download
forecast_window =48
negative = pd.read_csv(f"https://raw.githubusercontent.com/nabdeep-patel/flare_cme_association/refs/heads/main/Bobra%20et%20al.%202016/2024%20Analysis/Data%202024/negative{forecast_window}.csv")
positive = pd.read_csv(f"https://raw.githubusercontent.com/nabdeep-patel/flare_cme_association/refs/heads/main/Bobra%20et%20al.%202016/2024%20Analysis/Data%202024/positive{forecast_window}.csv")

data = pd.concat([positive,negative],axis = 0,ignore_index=True)

In [10]:
data

,Unnamed: 0,USFLUX,MEANGBT,MEANJZH,MEANPOT,SHRGT45,TOTUSJH,MEANGBH,MEANALP,MEANGAM,...,TOTPOT,MEANSHR,AREA_ACR,R_VALUE,ABSNJZH,HARPNUM,NOAA,Class,T_REC,Peak Time
0,0,1.080978e+22,120.026,0.028356,9200.471,47.881,1511.920,88.369,0.067207,59.424,...,2.340601e+23,47.746,470.031311,4.365,543.160,377,11158,X2.2,2011.02.13_01:56_TAI,2011-02-15 01:56:00
1,1,2.722415e+22,83.050,-0.001766,10861.630,41.395,1836.767,51.336,-0.003502,48.218,...,5.522075e+23,41.387,1038.552368,4.406,67.611,401,11166,M2.0,2011.03.05_14:30_TAI,2011-03-07 14:30:00
2,2,4.904070e+22,96.481,0.009289,11965.960,39.268,3761.582,55.250,0.018947,46.967,...,1.136452e+24,40.626,1895.631958,4.999,664.288,393,11164,M3.7,2011.03.05_20:12_TAI,2011-03-07 20:12:00
3,3,2.116432e+22,116.372,0.008077,5260.958,23.148,1593.884,57.156,0.024160,40.915,...,2.408346e+23,33.434,1168.906250,4.059,278.394,637,11226,M2.5,2011.06.05_06:41_TAI,2011-06-07 06:41:00
4,4,2.158570e+22,110.401,0.017384,13041.980,45.258,2859.943,77.675,0.041709,55.000,...,6.110265e+23,45.047,1350.126831,4.831,613.245,750,11261,M6.0,2011.08.01_13:48_TAI,2011-08-03 13:48:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1233,913,5.661549e+22,114.275,-0.014596,13663.580,37.498,5569.515,68.860,-0.025358,47.921,...,1.308115e+24,40.554,3710.337891,4.999,1052.163,12471,13936,M1.2,2024.12.28_17:30_TAI,2024-12-30 17:30:00.000
1234,914,5.671916e+22,114.178,-0.013650,13647.770,37.284,5594.463,68.815,-0.023761,47.815,...,1.309810e+24,40.549,3700.358154,5.024,986.438,12471,13936,M1.6,2024.12.28_17:42_TAI,2024-12-30 17:42:00.000
1235,915,5.667181e+22,114.730,-0.013281,13642.990,37.303,5658.353,68.536,-0.023007,47.537,...,1.302791e+24,40.322,3679.972656,5.025,954.948,12471,13936,M1.6,2024.12.28_18:24_TAI,2024-12-30 18:24:00.000
1236,916,5.685781e+22,114.334,-0.013176,13559.800,37.574,5709.022,68.309,-0.022897,47.720,...,1.304987e+24,40.542,3680.259521,5.025,954.802,12471,13936,M1.7,2024.12.28_18:33_TAI,2024-12-30 18:33:00.000


In [11]:
data["T_REC"] = pd.to_datetime(data["T_REC"], format="%Y.%m.%d_%H:%M_TAI")

In [13]:
data
data = data.reset_index(drop=True)

In [ ]:
import time
sharp_params = [
"USFLUX","MEANGAM","MEANGBT","MEANGBH","MEANGBZ",
"MEANJZD","TOTUSJZ","MEANALP","MEANJZH","TOTUSJH",
"ABSNJZH","SAVNCPP","MEANPOT","TOTPOT","MEANSHR","SHRGT45"
]

dataset = data.copy()

for p in sharp_params:
    dataset[p+"_PIL"] = np.nan

for i in range(data.shape[0]):
    time.sleep(1)
    print(f"Processing row {i}/{data.shape[0]}, Date:{data.loc[i, 'T_REC']}...")

    # Fetching the data from the database
    harpnum = int(data.loc[i, 'HARPNUM'])
    noaa_ar = data.loc[i, 'NOAA']
    date_time = data.loc[i, 'T_REC']
    date_time_str = date_time.strftime("%Y-%m-%d_%H-%M-%S")
    segments = ["Br","Bt","Bp","magnetogram","conf_disambig", "bitmap", "Bp_err", "Br_err", "Bt_err"]

    # Downloading the fits file
    query = Fido.search(a.jsoc.Time(date_time, date_time),
                       a.jsoc.Series("hmi.sharp_cea_720s"),
                       a.jsoc.PrimeKey("HARPNUM",harpnum),
                       *[a.jsoc.Segment(seg) for seg in segments],
                       a.jsoc.Notify("nabdeeppatel@gmail.com"))
    
    try:
        result = Fido.fetch(query, path=f"new_data_2016-2024/{forecast_window}/{harpnum}_{noaa_ar}_{date_time_str}")
    except Exception as e:
        print(f"Error at row {i}: {e}")
        continue

Processing row 500/838, Date:2024-05-07 21:21:00...


2026-06-19 03:30:20 - drms - INFO: Export request pending. [id=JSOC_20260618_003109, status=2]
2026-06-19 03:30:20 - drms - INFO: Waiting for 0 seconds...
2026-06-19 03:30:21 - drms - INFO: Export request pending. [id=JSOC_20260618_003109, status=1]
2026-06-19 03:30:21 - drms - INFO: Waiting for 5 seconds...
2026-06-19 03:30:26 - drms - INFO: Export request pending. [id=JSOC_20260618_003109, status=1]
2026-06-19 03:30:26 - drms - INFO: Waiting for 5 seconds...
2026-06-19 03:30:32 - drms - INFO: Export request pending. [id=JSOC_20260618_003109, status=1]
2026-06-19 03:30:32 - drms - INFO: Waiting for 5 seconds...
2026-06-19 03:30:38 - drms - INFO: Export request pending. [id=JSOC_20260618_003109, status=1]
2026-06-19 03:30:38 - drms - INFO: Waiting for 5 seconds...
2026-06-19 03:30:44 - drms - INFO: Export request pending. [id=JSOC_20260618_003109, status=1]
2026-06-19 03:30:44 - drms - INFO: Waiting for 5 seconds...
2026-06-19 03:30:50 - drms - INFO: Export request pending. [id=JSOC_20

INFO: 9 URLs found for download. Full request totaling 9MB [sunpy.net.jsoc.jsoc]


Files Downloaded: 100%|██████████| 9/9 [01:46<00:00, 11.84s/file]


Processing row 501/838, Date:2024-05-07 23:08:00...


...

2026-06-19 08:28:09 - drms - INFO: Export request pending. [id=JSOC_20260619_000565, status=2]
2026-06-19 08:28:09 - drms - INFO: Waiting for 0 seconds...
2026-06-19 08:28:10 - drms - INFO: Export request pending. [id=JSOC_20260619_000565, status=1]
2026-06-19 08:28:10 - drms - INFO: Waiting for 5 seconds...
2026-06-19 08:28:16 - drms - INFO: Export request pending. [id=JSOC_20260619_000565, status=1]
2026-06-19 08:28:16 - drms - INFO: Waiting for 5 seconds...
2026-06-19 08:28:21 - drms - INFO: Export request pending. [id=JSOC_20260619_000565, status=1]
2026-06-19 08:28:21 - drms - INFO: Waiting for 5 seconds...
2026-06-19 08:28:27 - drms - INFO: Export request pending. [id=JSOC_20260619_000565, status=1]
2026-06-19 08:28:27 - drms - INFO: Waiting for 5 seconds...
2026-06-19 08:28:33 - drms - INFO: Export request pending. [id=JSOC_20260619_000565, status=1]
2026-06-19 08:28:33 - drms - INFO: Waiting for 5 seconds...
2026-06-19 08:28:39 - drms - INFO: Export request pending. [id=JSOC_20

INFO: 9 URLs found for download. Full request totaling 23MB [sunpy.net.jsoc.jsoc]


Files Downloaded: 100%|██████████| 9/9 [02:57<00:00, 19.67s/file]


Processing row 622/838, Date:2024-08-01 18:39:00...


KeyboardInterrupt: 